# Exploratory Data Analysis: UCI Wine Recognition Dataset

**Week 1 Task — EDA + Visualisations**

**Dataset:** [Wine Recognition Dataset](https://archive.ics.uci.edu/dataset/109/wine) (UCI Machine Learning Repository), accessed via `sklearn.datasets.load_wine`.

**Goal:** Understand the data before modelling — clean it, explore relationships between features, and note takeaways that would shape any future model.

The dataset contains the results of a chemical analysis of 178 wines grown in the same region of Italy but derived from three different cultivars (classes). There are 13 continuous chemical measurements per wine (alcohol, acidity, phenols, color intensity, etc.).

## 1. Load & Inspect the Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

data = load_wine(as_frame=True)
df = data.frame
df['target'] = df['target'].map({0: 'class_0', 1: 'class_1', 2: 'class_2'})

df.head()

In [ ]:
df.shape

## 2. Cleaning

Before doing anything else, check for the usual suspects: missing values, duplicate rows, and data types.

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
print(df.dtypes)

**Result:** No missing values and no duplicate rows. The dataset is already tidy — all 13 chemical features are numeric (`float64`), and the target class has been mapped to readable labels. No imputation, type conversion, or de-duplication was needed here, which is unusual but does happen with well-curated UCI datasets.

Let's still look at summary statistics to get a feel for scale and spread across features.

In [ ]:
df.describe().T

**Observation:** Features are on very different scales — e.g. `proline` ranges up to ~1680 while `hue` maxes out around 1.7. This matters for any future modelling step (especially distance-based models like KNN or SVM), which would need feature scaling. It doesn't affect the visualisations below, but it's a key takeaway for later.

## 3. Visualisation 1 — Correlation Heatmap

Start broad: which chemical properties move together across the whole dataset?

In [ ]:
plt.figure(figsize=(11, 9))
corr = df.drop(columns='target').corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heatmap of Wine Chemical Properties")
plt.tight_layout()
plt.show()

**Takeaway:** `flavanoids` correlates strongly with `total_phenols` (r ≈ 0.86) and with `od280/od315_of_diluted_wines` (r ≈ 0.79). This makes chemical sense — flavanoids are a subclass of phenolic compounds. For modelling, these features carry overlapping information, so a model using all three may benefit from dimensionality reduction (e.g. PCA) or regularisation to avoid redundancy.

## 4. Visualisation 2 — Flavanoids vs Total Phenols by Class

Since these two are the most correlated pair, check whether that relationship also separates the wine classes — a strong signal for whether these features would help a classifier.

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="total_phenols", y="flavanoids", hue="target", palette="Set2", s=60)
plt.title("Flavanoids vs Total Phenols by Wine Class")
plt.xlabel("Total Phenols")
plt.ylabel("Flavanoids")
plt.tight_layout()
plt.show()

**Takeaway:** The three classes form fairly distinct clusters along this relationship, with `class_2` sitting clearly lower on both axes and `class_0`/`class_1` more separated along the diagonal. This pair alone looks like it would carry real predictive power for classifying wine cultivar — a strong candidate feature pair for a future model.

## 5. Visualisation 3 — Boxplots for Outlier Detection

Check four features that showed high variance in the summary stats for outliers, using the IQR method.

In [ ]:
cols_to_check = ['malic_acid', 'proanthocyanins', 'color_intensity', 'proline']

plt.figure(figsize=(12, 6))
df_melt = df[cols_to_check].melt(var_name='feature', value_name='value')
sns.boxplot(data=df_melt, x='feature', y='value', hue='feature', palette="Set3", legend=False)
plt.title("Boxplots for Outlier Detection in Key Features")
plt.tight_layout()
plt.show()

print("Outlier counts (IQR method, 1.5x rule):")
for col in cols_to_check:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
    print(f"  {col}: {len(outliers)} outlier(s)")

**Takeaway:** `color_intensity` has the most outliers (4), followed by `malic_acid` (3) and `proanthocyanins` (2); `proline` has none despite its huge scale — it's just naturally spread out, not skewed by extreme points. The outliers here are mild and few, likely genuine chemical variation rather than data errors, so I would not drop them outright. Still worth flagging: a model sensitive to outliers (e.g. linear regression, KNN) might want winsorizing or robust scaling on `color_intensity` and `malic_acid`.

## 6. Visualisation 4 — Alcohol Distribution by Class

Check whether a simpler, single feature like alcohol content already separates wine classes.

In [ ]:
plt.figure(figsize=(8, 6))
sns.histplot(data=df, x="alcohol", hue="target", kde=True, palette="Set2", element="step")
plt.title("Alcohol Content Distribution by Wine Class")
plt.xlabel("Alcohol (%)")
plt.tight_layout()
plt.show()

**Takeaway:** `class_1` skews noticeably lower in alcohol content than `class_0` and `class_2`, whose distributions overlap more with each other. Alcohol alone would partially separate `class_1` from the rest, but wouldn't reliably distinguish `class_0` from `class_2` — confirming that a real classifier will need to combine multiple features rather than rely on any single one.

## 7. Visualisation 5 — OD280/OD315 vs Proline by Class

Check a second promising feature pair, since `od280/od315_of_diluted_wines` also showed up in the correlation heatmap.

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x="od280/od315_of_diluted_wines", y="proline", hue="target", palette="Set2", s=60)
plt.title("OD280/OD315 vs Proline by Wine Class")
plt.xlabel("OD280/OD315 of Diluted Wines")
plt.ylabel("Proline")
plt.tight_layout()
plt.show()

**Takeaway:** This pair separates the classes even more cleanly than the first scatter plot — `class_0` stands out with high proline, while `class_1` and `class_2` separate mainly along the OD280/OD315 axis. This is likely to be one of the more powerful feature combinations for a downstream classification model.

## 8. Summary of Key Takeaways

1. **Data quality:** No missing values or duplicates — the dataset was clean out of the box. The only prep a model would need is **feature scaling**, since features span very different ranges (e.g. `proline` ~278–1680 vs `hue` ~0.5–1.7).
2. **Strong correlations:** `flavanoids`, `total_phenols`, and `od280/od315_of_diluted_wines` are highly correlated (r up to 0.86) — chemically sensible, but a signal to consider dimensionality reduction or feature selection later to avoid redundant information.
3. **Class separation:** Two feature pairs — (`total_phenols`, `flavanoids`) and (`od280/od315_of_diluted_wines`, `proline`) — visually separate the three wine cultivars quite well, suggesting a classifier trained on this data should perform strongly even with a fairly simple model.
4. **Outliers are mild:** `color_intensity`, `malic_acid`, and `proanthocyanins` have a handful of outliers (2–4 each) via the IQR method. These look like genuine variation rather than data errors, but robust scaling is worth testing if outlier-sensitive models are used.
5. **No single feature dominates:** Alcohol content alone separates `class_1` from the others but not `class_0` from `class_2` — reinforcing that a future model will need multiple features together, not just univariate rules.

**Next steps for modelling:** scale features (StandardScaler), consider PCA given the correlated feature clusters, and expect a multi-feature classifier (e.g. logistic regression, random forest) to perform well given how cleanly some feature pairs already separate the classes.